## 03 · Evaluate

Scores the traces from `02_run_pipeline` offline with `mlflow.genai.evaluate(data=<traces>)` and no `predict_fn`. Feedback is attached to the same traces, so opening one in the experiment shows its transcript, tool calls, scores and rationales together.

| Stage | Question | Scorers |
| --- | --- | --- |
| 1 ASR | Did it hear the question correctly? | WER, domain-term error rate, number accuracy, exact match (code) |
| 2 Reasoning | Given the true transcript (2A) and what it heard (2B), did it answer and use tools correctly? | tool name and argument match, numeric answer, spoken suitability (code); correctness and voice guidelines (LLM judges) |
| 3 TTS | Did it say the answer correctly? On the reference answer (3A, same text for every candidate) and on its own answer (3B) | round-trip WER, term and number accuracy, speaking rate, completeness (code) |

**ASR impact** = score(2A, true transcript) minus score(2B, as heard): the ASR error carried into reasoning. Comparing 3A across candidates compares the voices alone; 3B adds the effect of how each candidate phrases its answers.

**Illustrative data.** Input audio is synthetic, which flatters ASR. Round-trip ASR measures intelligibility, not naturalness.

In [ ]:
%pip install -q "mlflow[databricks]>=3.15"
dbutils.library.restartPython()

In [ ]:
import json, os, sys, time

sys.path.insert(0, os.path.abspath(".."))
import mlflow
from IPython.display import Audio, Markdown, display as show
from voice_evals import config as C
from voice_evals.delta import replace_run
from voice_evals.evaluate import (asr_impact, attribution, collect_results, load_run_traces, overview,
                                  run_evaluations, scorecard, worst)

dbutils.widgets.text("run_id", "", "Run ID (blank = latest)")
RUN_ID = dbutils.widgets.get("run_id").strip() or (
    spark.sql(f"SELECT max(run_id) AS r FROM {C.TABLES['pipeline_outputs']}").first()["r"])
experiment = C.set_experiment()
print(f"Evaluating run {RUN_ID} in {experiment.name} ({experiment.experiment_id}), judge {C.JUDGE_MODEL}")

In [ ]:
traces = load_run_traces(experiment.experiment_id, RUN_ID)
print(f"{len(traces)} traces")
eval_runs = run_evaluations(traces, run_id=RUN_ID, judge_model=C.JUDGE_MODEL)
eval_runs

In [ ]:
for _ in range(12):  # feedback on UC-stored traces can take a few seconds to show up
    results = collect_results(experiment.experiment_id, RUN_ID)
    if len(results) and results["trace_id"].nunique() >= len(traces):
        break
    time.sleep(10)

pipeline = spark.table(C.TABLES["pipeline_outputs"]).where(f"run_id = '{RUN_ID}'").toPandas()
card = scorecard(results, run_id=RUN_ID, dataset_version=C.DATASET_VERSION)
attr = attribution(results).assign(run_id=RUN_ID)
replace_run(spark, C.TABLES["eval_results"], "eval_results", results, RUN_ID)
replace_run(spark, C.TABLES["eval_scorecard"], "eval_scorecard", card, RUN_ID)
replace_run(spark, C.TABLES["attribution"], "attribution", attr, RUN_ID)
print(f"{len(results)} assessments on {results['trace_id'].nunique()} traces; scorecard rows: {len(card)}")

In [ ]:
summary = overview(results, pipeline)
display(summary)

In [ ]:
display(asr_impact(results))

In [ ]:
display(attr.sort_values(["attribution", "candidate", "utterance_id"]))
display(attr.groupby(["candidate", "attribution"]).size().unstack(fill_value=0).reset_index())

In [ ]:
golden_audio = {r["utterance_id"]: r["audio_uri"] for r in spark.table(C.TABLES["golden_utterances"])
               .where(f"dataset_version = '{C.DATASET_VERSION}'").collect()}
heard = pipeline[pipeline["condition"] == "as_heard"].set_index(["candidate", "utterance_id"])

def show_worst(metric, n=3):
    show(Markdown(f"### Worst {n}: `{metric}`"))
    for _, r in worst(results, metric, n).iterrows():
        row = heard.loc[(r["candidate"], r["utterance_id"])]
        show(Markdown(f"**{r['candidate']} · {r['utterance_id']}** ({r['category']}): {metric} = {r['value']}  \n"
                      f"{(r['rationale'] or '').replace(chr(10), '  ' + chr(10))}  \ntrace `{r['trace_id']}`"))
        show(Markdown("input audio"), Audio(filename=golden_audio[r["utterance_id"]]))
        if isinstance(row.get("tts_uri"), str):
            show(Markdown("spoken answer"), Audio(filename=row["tts_uri"]))

show_worst("asr_wer")
show_worst("tool_args_match")
show_worst("tts_roundtrip_wer")

In [ ]:
lines = [f"## Voice Agent Evals, run `{RUN_ID}` (Illustrative data)", ""]
for _, r in summary.iterrows():
    counts = attr[attr["candidate"] == r["candidate"]]["attribution"].value_counts().to_dict()
    lines.append(
        f"* **{r['candidate']}**: Stage 1 WER {r['S1 WER']:.2f}, term error {r['S1 term err']:.2f}. "
        f"Stage 2 correct {r['S2A correct']:.2f} on the true transcript vs {r['S2B correct']:.2f} as heard "
        f"(ASR impact {r['ASR impact (correct)']:+.2f}); tools {r['S2A tool match']:.2f} vs {r['S2B tool match']:.2f}. "
        f"Stage 3 round-trip WER {r['S3 round-trip WER (reference)']:.2f} on the reference answer, "
        f"{r['S3 round-trip WER']:.2f} on its own answer. Turns: {counts}.")
lines += ["", f"Judge: `{C.JUDGE_MODEL}`. Reference ASR: `{C.REFERENCE_ASR}`. Input audio is synthetic."]
show(Markdown("\n".join(lines)))

In [ ]:
dbutils.notebook.exit(json.dumps({"run_id": RUN_ID, "traces": len(traces), "assessments": len(results),
                                  "overview": summary.to_dict("records")}, default=str))